In [4]:
# Copyright (c) 2024 Microsoft Corporation.
# Licensed under the MIT License.


In [5]:
import os

import pandas as pd
import tiktoken

from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.indexer_adapters import (
    read_indexer_covariates,
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.embedding import OpenAIEmbedding
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.question_gen.local_gen import LocalQuestionGen
from graphrag.query.structured_search.local_search.mixed_context import (
    LocalSearchMixedContext,
)
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag.vector_stores.lancedb import LanceDBVectorStore


## Local Search Example

Local search method generates answers by combining relevant data from the AI-extracted knowledge-graph with text chunks of the raw documents. This method is suitable for questions that require an understanding of specific entities mentioned in the documents (e.g. What are the healing properties of chamomile?).

### Load text units and graph data tables as context for local search

- In this test we first load indexing outputs from parquet files to dataframes, then convert these dataframes into collections of data objects aligning with the knowledge model.

### Load tables to dataframes

In [6]:
COMMUNITY_REPORT_TABLE = "output/create_final_community_reports.parquet"
ENTITY_TABLE = "output/create_final_nodes.parquet"
ENTITY_EMBEDDING_TABLE = "output/create_final_entities.parquet"
RELATIONSHIP_TABLE = "output/create_final_relationships.parquet"
COVARIATE_TABLE = "output/create_final_covariates.parquet"
TEXT_UNIT_TABLE = "output/create_final_text_units.parquet"
COMMUNITY_LEVEL = 2

COMMUNITY_TABLE = "output/create_final_communities.parquet"


In [ ]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
from io import BytesIO

# Azure Blob Storage credentials
# Azure Blob Storage credentials
connection_string = "your_connection_string_here"
container_name = "output"

# Initialize Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

def read_parquet_from_azure(file_path):
    """Reads a Parquet file from Azure Blob Storage and returns a Pandas DataFrame."""
    try:
        blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_path)
        parquet_data = blob_client.download_blob().readall()
        df = pd.read_parquet(BytesIO(parquet_data))
        print(f"✅ Successfully read: {file_path}")
        return df
    except Exception as e:
        print(f"❌ Error reading {file_path}: {e}")
        return None  # Return None if the file is not found


#### Read entities

In [ ]:
# read nodes table to get community and degree data
entity_df = read_parquet_from_azure(ENTITY_TABLE)
entity_embedding_df = read_parquet_from_azure(ENTITY_EMBEDDING_TABLE)

entities = read_indexer_entities(entity_df, entity_embedding_df, COMMUNITY_LEVEL)

# load description embeddings to an in-memory lancedb vectorstore
# to connect to a remote db, specify url and port values.
# description_embedding_store = LanceDBVectorStore(
#     collection_name="entity_description_embeddings",
# )
# description_embedding_store.connect(db_uri=LANCEDB_URI)
# entity_description_embeddings = store_entity_semantic_embeddings(
#     entities=entities, vectorstore=description_embedding_store
# )


from graphrag.vector_stores.azure_ai_search import AzureAISearchVectorStore

# to connect to a remote db, specify url and port values.
description_embedding_store = AzureAISearchVectorStore(collection_name="default-community-full_content",)
description_embedding_store.connect(url = " ",api_key =" " )



print(f"Entity count: {len(entity_df)}")
entity_df.head()


✅ Successfully read: output/create_final_nodes.parquet
✅ Successfully read: output/create_final_entities.parquet
Entity count: 127


,id,human_readable_id,title,community,level,degree,x,y
0,df671504-b4f4-4902-9b63-2de11870e1b1,0,MICROSOFT,2,0,2,0.0,0.0
1,df671504-b4f4-4902-9b63-2de11870e1b1,0,MICROSOFT,11,1,2,0.0,0.0
2,99cfec08-f072-47e8-a1ec-5de2e4204a87,1,DRAM,2,0,9,0.0,0.0
3,99cfec08-f072-47e8-a1ec-5de2e4204a87,1,DRAM,8,1,9,0.0,0.0
4,d7975622-0be1-472a-b013-56c7d5a93890,2,NAND,2,0,4,0.0,0.0


#### Read relationships

In [9]:
relationship_df = read_parquet_from_azure(RELATIONSHIP_TABLE)
relationships = read_indexer_relationships(relationship_df)

print(f"Relationship count: {len(relationship_df)}")
relationship_df.head()


✅ Successfully read: output/create_final_relationships.parquet
Relationship count: 101


,id,human_readable_id,source,target,description,weight,combined_degree,text_unit_ids
0,1d9ffd18-eda7-4b4c-b1ce-518e19bd997d,0,MICROSOFT,DRAM,Microsoft is specifically mentioned in relatio...,9.0,11,[6b3b522914718e5090c120914dc743c783893451c735c...
1,d05e92b9-16f2-48e1-84b8-bb3e7f910853,1,DRAM,Q423,DRAM pricing and market strategies are discuss...,8.0,14,[6b3b522914718e5090c120914dc743c783893451c735c...
2,15ca7bbf-00e1-4ae5-ad74-5e681c684111,2,NAND,Q423,NAND price increases are discussed for the fou...,8.0,9,[6b3b522914718e5090c120914dc743c783893451c735c...
3,a7352fa7-f898-408d-a478-a6c445345dd0,3,Q424,FREE CASH FLOW,Improvements in free cash flow reduce cash flo...,7.0,4,[6b3b522914718e5090c120914dc743c783893451c735c...
4,a123796e-93d3-4a64-8353-9037b93f8330,4,2024,SAMSUNG,Samsung's production strategies for 2024 are d...,7.0,18,[6b3b522914718e5090c120914dc743c783893451c735c...


In [10]:
covariate_df = read_parquet_from_azure(COVARIATE_TABLE)

claims = read_indexer_covariates(covariate_df)

print(f"Claim records: {len(claims)}")
covariates = {"claims": claims}


✅ Successfully read: output/create_final_covariates.parquet
Claim records: 37


#### Read community reports

In [11]:
report_df = read_parquet_from_azure(COMMUNITY_REPORT_TABLE)
reports = read_indexer_reports(report_df, entity_df, COMMUNITY_LEVEL)

print(f"Report records: {len(report_df)}")
report_df.head()


✅ Successfully read: output/create_final_community_reports.parquet
Report records: 20


,id,human_readable_id,community,parent,level,title,summary,full_content,rank,rank_explanation,findings,full_content_json,period,size
0,e126e37e043a4c03b8ee01a915bffcdb,8,8,2,1,DRAM Market Dynamics and Relationships in 2023,The community focuses on the Dynamic Random-Ac...,# DRAM Market Dynamics and Relationships in 20...,7.5,The impact severity rating is high due to the ...,[{'explanation': 'Dynamic Random-Access Memory...,"{\n ""title"": ""DRAM Market Dynamics and Rela...",2025-02-16,5
1,39cd6c18fb7d40c6bfeec400e53a4d53,9,9,2,1,China Mobile OEMs and Micron: Impact of NAND a...,This report examines the interconnected relati...,# China Mobile OEMs and Micron: Impact of NAND...,7.5,The impact severity rating is high due to the ...,"[{'explanation': 'China Mobile OEMs, which man...","{\n ""title"": ""China Mobile OEMs and Micron:...",2025-02-16,4
2,2a73f471ca4949c7b1217bc81911be56,10,10,2,1,Q423 and 64GB RD4 Market Analysis,The community focuses on the interactions betw...,# Q423 and 64GB RD4 Market Analysis\n\nThe com...,6.5,The impact severity rating is moderately high ...,[{'explanation': 'Q423 is identified as a cruc...,"{\n ""title"": ""Q423 and 64GB RD4 Market Anal...",2025-02-16,2
3,d46b86dad1934d779a78ab9947d21fec,11,11,2,1,Microsoft and De Dios & Associates Market Anal...,This report focuses on the relationship betwee...,# Microsoft and De Dios & Associates Market An...,6.5,The impact severity rating is moderately high ...,[{'explanation': 'Microsoft is a central figur...,"{\n ""title"": ""Microsoft and De Dios & Assoc...",2025-02-16,2
4,0fc8ce24364e41e293e077b9203064e9,12,12,2,1,HBMX and TSV Technology Competition,The community focuses on the competition betwe...,# HBMX and TSV Technology Competition\n\nThe c...,6.5,The impact severity rating is moderately high ...,"[{'explanation': 'HBMX, or High Bandwidth Memo...","{\n ""title"": ""HBMX and TSV Technology Compe...",2025-02-16,2


#### Read text units

In [12]:
text_unit_df = read_parquet_from_azure(TEXT_UNIT_TABLE)
text_units = read_indexer_text_units(text_unit_df)

print(f"Text unit records: {len(text_unit_df)}")
text_unit_df.head()


✅ Successfully read: output/create_final_text_units.parquet
Text unit records: 10


,id,human_readable_id,text,n_tokens,document_ids,entity_ids,relationship_ids,covariate_ids
0,6b3b522914718e5090c120914dc743c783893451c735c1...,1,"[\n {\n ""MarketProvider"": ""De Dios & Assoc...",1200,[16e284a3992a0a339593c7161f0d5f9f7d947a847f32f...,"[df671504-b4f4-4902-9b63-2de11870e1b1, 99cfec0...","[1d9ffd18-eda7-4b4c-b1ce-518e19bd997d, d05e92b...","[e4aaa5ee-6fbe-4cab-a5aa-14a2237b7d20, c42c123..."
1,70c5de1382208c2d0e28761407c9e2f840a9eba5f193db...,2,"premium, new Intel next-gen platformsCustomer...",1200,[16e284a3992a0a339593c7161f0d5f9f7d947a847f32f...,"[df671504-b4f4-4902-9b63-2de11870e1b1, 99cfec0...","[5abae5b0-b30c-49e1-9720-1ee43f90579d, 89cffcb...","[e6551d82-11c2-40cb-8c4d-1f8efaa1e9fa, c242b4b..."
2,f0c033042945d0bc9c95dab6c89820ee45e42c7c373047...,3,"14"",\n ""Content"": ""10/30/202314How High Can...",677,[16e284a3992a0a339593c7161f0d5f9f7d947a847f32f...,"[e0f37ede-9b49-44bf-8121-b2173a09be34, e3b32ce...","[5f01316f-e25c-4e72-a668-2f32ad153ddf, b4a14fe...",[2e904820-566c-497b-aea8-3f1f28ea5dfb]
3,dceec340e7a10a404359735062ab9370e97372a4ddcc97...,4,"[\n {\n ""MarketProvider"": ""Cleveland Resea...",1200,[63c8709f1f8626af2f6cefb3763ba77f8e0da721be8f4...,"[99cfec08-f072-47e8-a1ec-5de2e4204a87, d797562...","[6d1a7080-22be-4821-a216-53ad1178fd69, 84d5768...","[d732490f-e9cb-420a-bfee-1ef16b331330, 2c24c2e..."
4,a505786dabf364ac26f20c98d78b36b6de199da6ab8b9b...,5,% |Source: Cleveland Research- 3. DRAM ASPs ...,1200,[63c8709f1f8626af2f6cefb3763ba77f8e0da721be8f4...,"[e7eff97c-f90d-4241-97d7-fa7824fc6d0c, 4b19655...","[6887de94-8c4c-43b6-886f-c26c91e64ce1, 92449ca...","[5f3154ea-4a3f-4163-9ee4-3a2a70093818, 0aedafa..."


In [17]:
import json 
# Load configuration from config.json
with open("C:/Users/dipankar.nath/Downloads/Graphrag/pro_code/config.json") as config_file:
    config = json.load(config_file)

api_key =  config["llm"]["api_key"]
llm_model =  config["llm"]["engine"]
azure_endpoint = config["llm"]["azure_endpoint"]
api_version= config["llm"]["api_version"]

embedding_model =  config["embed"]["model"]
embed_endpoint = config["embed"]["azure_endpoint"]
embed_version= config["embed"]["api_version"]

llm = ChatOpenAI(
    api_key=api_key,
    api_base =azure_endpoint,
    api_version=api_version,
    model=llm_model,
    api_type=OpenaiApiType.AzureOpenAI,  # OpenaiApiType.OpenAI or OpenaiApiType.AzureOpenAI
    max_retries=20,
)

token_encoder = tiktoken.get_encoding("cl100k_base")

text_embedder = OpenAIEmbedding(
    api_key=api_key,
    api_base=embed_endpoint,
    api_type=OpenaiApiType.AzureOpenAI,
    model=embedding_model,
    deployment_name=embedding_model,
    api_version=embed_version,
    max_retries=20,
)


### Create local search context builder

In [18]:
context_builder = LocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    covariates=covariates,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.ID,  # if the vectorstore uses entity title as ids, set this to EntityVectorStoreKey.TITLE
    text_embedder=text_embedder,
    token_encoder=token_encoder,
)


### Create local search engine

In [34]:
# text_unit_prop: proportion of context window dedicated to related text units
# community_prop: proportion of context window dedicated to community reports.
# The remaining proportion is dedicated to entities and relationships. Sum of text_unit_prop and community_prop should be <= 1
# conversation_history_max_turns: maximum number of turns to include in the conversation history.
# conversation_history_user_turns_only: if True, only include user queries in the conversation history.
# top_k_mapped_entities: number of related entities to retrieve from the entity description embedding store.
# top_k_relationships: control the number of out-of-network relationships to pull into the context window.
# include_entity_rank: if True, include the entity rank in the entity table in the context window. Default entity rank = node degree.
# include_relationship_weight: if True, include the relationship weight in the context window.
# include_community_rank: if True, include the community rank in the context window.
# return_candidate_context: if True, return a set of dataframes containing all candidate entity/relationship/covariate records that
# could be relevant. Note that not all of these records will be included in the context window. The "in_context" column in these
# dataframes indicates whether the record is included in the context window.
# max_tokens: maximum number of tokens to use for the context window.


local_context_params = {
    "text_unit_prop": 0.5,
    "community_prop": 0.1,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 10,
    "top_k_relationships": 10,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": False,
    "return_candidate_context": False,
    "embedding_vectorstore_key": EntityVectorStoreKey.ID,  # set this to EntityVectorStoreKey.TITLE if the vectorstore uses entity title as ids
    "max_tokens": 12_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
}

llm_params = {
    "max_tokens": 8_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 1000=1500)
    "temperature": 0.0,
}


In [35]:
search_engine = LocalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="multiple paragraphs",  # free form text describing the response type and format, can be anything, e.g. prioritized list, single paragraph, multiple paragraphs, multiple-page report
)


### Run local search on sample queries

In [ ]:
question = "When is the supplier inventory for DRAMs expected to be the highest?"
result = await search_engine.asearch(question)
print(result.response)


In [ ]:
print(result)


SearchResult(response='The supplier inventory for DRAMs is expected to be the highest in Q3 2023. This projection is based on the data indicating a steady increase in inventory levels from Q1 2023 through Q3 2023. The inventory levels are forecasted to peak in Q3 before slightly declining in Q4 2023.\n\n### Inventory Trends Overview\n- **Q1 2023**: The inventory levels begin to rise, indicating an increase in production or a decrease in demand.\n- **Q2 2023**: The upward trend continues, suggesting ongoing adjustments in supply chain management or market conditions.\n- **Q3 2023**: This quarter is expected to see the highest inventory levels, which could be due to anticipation of market demand or strategic stockpiling.\n- **Q4 2023**: There is a slight reduction in inventory levels, possibly due to increased sales or adjustments in production strategies.\n\n### Implications\nHigh inventory levels in Q3 2023 might suggest that suppliers are preparing for a seasonal increase in demand, o

#### Inspecting the context data used to generate the response

In [26]:
%pip install yfiles_jupyter_graphs --quiet
from yfiles_jupyter_graphs import GraphWidget


# converts the entities dataframe to a list of dicts for yfiles-jupyter-graphs
def convert_entities_to_dicts(df):
    """Convert the entities dataframe to a list of dicts for yfiles-jupyter-graphs."""
    nodes_dict = {}
    for _, row in df.iterrows():
        # Create a dictionary for each row and collect unique nodes
        node_id = row["title"]
        if node_id not in nodes_dict:
            nodes_dict[node_id] = {
                "id": node_id,
                "properties": row.to_dict(),
            }
    return list(nodes_dict.values())


# converts the relationships dataframe to a list of dicts for yfiles-jupyter-graphs
def convert_relationships_to_dicts(df):
    """Convert the relationships dataframe to a list of dicts for yfiles-jupyter-graphs."""
    relationships = []
    for _, row in df.iterrows():
        # Create a dictionary for each row
        relationships.append({
            "start": row["source"],
            "end": row["target"],
            "properties": row.to_dict(),
        })
    return relationships


w = GraphWidget()
w.directed = True
w.nodes = convert_entities_to_dicts(entity_df)
w.edges = convert_relationships_to_dicts(relationship_df)


Note: you may need to restart the kernel to use updated packages.


c:\Users\dipankar.nath\Downloads\graphrag_custom\.venv\Lib\site-packages\jupyter_client\session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


In [30]:
# show title on the node
w.node_label_mapping = "title"


# map community to a color
def community_to_color(community):
    """Map a community to a color."""
    colors = [
        "crimson",
        "darkorange",
        "indigo",
        "cornflowerblue",
        "cyan",
        "teal",
        "green",
    ]
    return (
        colors[int(community) % len(colors)] if community is not None else "lightgray"
    )


def edge_to_source_community(edge):
    """Get the community of the source node of an edge."""
    source_node = next(
        (entry for entry in w.nodes if entry["properties"]["title"] == edge["start"]),
        None,
    )
    source_node_community = source_node["properties"]["community"]
    return source_node_community if source_node_community is not None else None


w.node_color_mapping = lambda node: community_to_color(node["properties"]["community"])
w.edge_color_mapping = lambda edge: community_to_color(edge_to_source_community(edge))
# map size data to a reasonable factor
w.node_scale_factor_mapping = lambda node: 0.5 + node["properties"]["size"] * 1.5 / 20
# use weight for edge thickness
w.edge_thickness_factor_mapping = "weight"


In [ ]:
"""
Helper function to visualize the result context with `yfiles-jupyter-graphs`.

The dataframes are converted into supported nodes and relationships lists and then passed to yfiles-jupyter-graphs.
Additionally, some values are mapped to visualization properties.
"""


def show_graph(result):
    """Visualize the result context with yfiles-jupyter-graphs."""
    from yfiles_jupyter_graphs import GraphWidget

    if (
        "entities" not in result.context_data
        or "relationships" not in result.context_data
    ):
        msg = "The passed results do not contain 'entities' or 'relationships'"
        raise ValueError(msg)

    # converts the entities dataframe to a list of dicts for yfiles-jupyter-graphs
    def convert_entities_to_dicts(df):
        """Convert the entities dataframe to a list of dicts for yfiles-jupyter-graphs."""
        nodes_dict = {}
        for _, row in df.iterrows():
            # Create a dictionary for each row and collect unique nodes
            node_id = row["entity"]
            if node_id not in nodes_dict:
                nodes_dict[node_id] = {
                    "id": node_id,
                    "properties": row.to_dict(),
                }
        return list(nodes_dict.values())

    # converts the relationships dataframe to a list of dicts for yfiles-jupyter-graphs
    def convert_relationships_to_dicts(df):
        """Convert the relationships dataframe to a list of dicts for yfiles-jupyter-graphs."""
        relationships = []
        for _, row in df.iterrows():
            # Create a dictionary for each row
            relationships.append({
                "start": row["source"],
                "end": row["target"],
                "properties": row.to_dict(),
            })
        return relationships

    w = GraphWidget()
    # use the converted data to visualize the graph
    w.nodes = convert_entities_to_dicts(result.context_data["entities"])
    w.edges = convert_relationships_to_dicts(result.context_data["relationships"])
    w.directed = True
    # show title on the node
    w.node_label_mapping = "entity"
    # use weight for edge thickness
    w.edge_thickness_factor_mapping = "weight"
    display(w)


show_graph(result)


ValueError: The passed results do not contain 'entities' or 'relationships'